# BIL545 - Proje 1
## Banka Pazarlamasında Abonelik Tahmini

### Bölüm 1 - Veri, Ön İşleme ve Veri Sızıntısı

Bu çalışmada UCI Bank Marketing veri kümesi
kullanılarak müşterilerin vadeli mevduata abonelik
durumları incelenecektir.

Amaç, veri kümesini tanımak, veri ön işleme
yöntemlerini değerlendirmek ve veri sızıntısının
model sonuçlarına etkisini araştırmaktır.

In [1]:
import sys
import random
import numpy as np
import pandas as pd
import sklearn
import scipy

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Random seed:", SEED)

Python: 3.13.18
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1
SciPy: 1.16.3
Random seed: 42


## 1. Veri Kümesinin Yüklenmesi

Bu projede UCI Machine Learning Repository üzerinden
sağlanan Bank Marketing veri kümesi kullanılmaktadır.

Veri kümesi, Portekiz'deki bir bankanın telefon
pazarlama kampanyalarına ait kayıtları içermektedir.

Hedef değişken `y`, müşterinin vadeli mevduata
abone olup olmadığını göstermektedir.

Veri dosyası GitHub deposuna eklenmeyecek,
Google Colab ortamına ayrıca yüklenecektir.

In [2]:
from pathlib import Path
from google.colab import files

# Veri dosyasının konumu
dosya_yolu = Path("/content/bank-full.csv")

# Dosya bulunmuyorsa kullanıcıdan yüklemesini iste
if not dosya_yolu.exists():
    print("Lütfen bank-full.csv dosyasını yükleyin.")
    files.upload()

# Dosyanın varlığını kontrol et
if not dosya_yolu.exists():
    raise FileNotFoundError("bank-full.csv bulunamadı.")

# Veri kümesini oku
df = pd.read_csv(dosya_yolu, sep=";")

# Temel bilgileri göster
print("Satır sayısı:", df.shape[0])
print("Sütun sayısı:", df.shape[1])
print("Sütunlar:", df.columns.tolist())

display(df.head())

Lütfen bank-full.csv dosyasını yükleyin.


Saving bank-full.csv to bank-full.csv
Satır sayısı: 45211
Sütun sayısı: 17
Sütunlar: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Keşifsel Veri Analizi (EDA)

Bu bölümde veri kümesinin yapısı, değişken türleri,
hedef değişkenin dağılımı ve yinelenen kayıtlar
incelenecektir.

Amaç, model geliştirmeden önce verinin temel
özelliklerini ve olası problemlerini belirlemektir.

In [3]:
# Veri kümesinin boyutları
print("Veri kümesinin boyutu:", df.shape)

# Değişkenlerin veri tipleri
print("\nDeğişken tipleri:")
display(df.dtypes.to_frame(name="Veri tipi"))

# Hedef değişkenin dağılımı
print("\nHedef değişken dağılımı:")

hedef_dagilimi = pd.DataFrame({
    "Adet": df["y"].value_counts(),
    "Oran (%)": df["y"].value_counts(normalize=True) * 100
})

display(hedef_dagilimi.round(2))

# Yinelenen satır kontrolü
print("\nYinelenen satır sayısı:", df.duplicated().sum())

Veri kümesinin boyutu: (45211, 17)

Değişken tipleri:


,Veri tipi
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64



Hedef değişken dağılımı:


,Adet,Oran (%)
y,,
no,39922,88.3
yes,5289,11.7



Yinelenen satır sayısı: 0


### Yorum - İlk Veri İncelemesi

Veri kümesinde 45.211 gözlem ve 17 değişken bulunmaktadır.
Bunların 16'sı girdi değişkeni, biri hedef değişken olan
y sütunudur.

Değişkenlerin 7'si int64, 10'u object veri tipindedir.
Ancak bu tipler değişkenlerin ölçüm ölçeklerini
doğrudan göstermemektedir.

Hedef değişken incelendiğinde kayıtların %88,30'unun
abonelikle sonuçlanmadığı, %11,70'inin ise abonelikle
sonuçlandığı görülmektedir.

Bu dağılım, sınıflar arasında belirgin bir dengesizlik
olduğunu göstermektedir. Bu nedenle model
değerlendirmesinde yalnızca doğruluk (accuracy)
ölçütü yeterli olmayacaktır.

Veri kümesinde birebir yinelenen satır bulunmamıştır.
Bununla birlikte müşteri kimlikleri mevcut olmadığından
aynı müşterinin farklı kampanya kayıtlarında
bulunup bulunmadığı belirlenememektedir.